## tl;dr

Within the 169 current-method one-on-one SLUGFESTER scorecards that contain one published theistic/religious side and one published non-theistic/skeptical side, the non-theistic side averages **83.79** versus **77.46**, a **6.34-point** advantage. It receives the higher overall score in **144 debates**, ties in **5**, and scores lower in **20**. The result is strong evidence of a pattern in this catalogue, not evidence that non-theism is true or that the pattern would generalize to a representative external debate population.

## Context & Methods

The unit is one published dyadic debate scorecard. The classification follows the position presented by each side, not a participant's private beliefs. Team and panel debates use a different approximate method and are excluded. The companion script contains the explicit 210-row inclusion taxonomy, derives all results from `src/data/debates.js`, and writes `results.json` plus `taxonomy.csv`.

### Key Assumptions

- A side must present a recognizable theistic/religious case or a recognizable non-theistic/skeptical case.
- Debates that are merely religion-adjacent, have two religious sides, or do not compare these positions are excluded.
- A positive margin means the non-theistic side scored higher.
- Resampling intervals describe stability within this catalogue; the catalogue is not a random sample.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

candidate = Path('docs/analysis/non-theist-vs-theist-2026-09-01')
analysis_dir = candidate.resolve() if candidate.exists() else Path.cwd().resolve()
completed = subprocess.run(
    [sys.executable, str(analysis_dir / 'analysis.py')],
    check=True, capture_output=True, text=True,
)
results = json.loads((analysis_dir / 'results.json').read_text())
quality = results['data_quality']
print(
    f"Analysis executed successfully: {quality['published_debates']} published debates, "
    f"{quality['current_method_dyadic_debates']} current-method dyads, "
    f"{quality['classified_comparisons']} classified comparisons."
)

Analysis executed successfully: 226 published debates, 210 current-method dyads, 169 classified comparisons.


## Data

The source profile checks catalogue coverage, common-method eligibility, uniqueness, missingness, and score ranges before any stance comparison is calculated.

In [2]:
for key, value in quality.items():
    if key != 'overall_score_range':
        print(f"{key:55s} {value:>6}")

published_debates                                          226
current_method_dyadic_debates                              210
team_or_panel_debates_excluded                              16
classified_comparisons                                     169
current_method_dyadic_debates_excluded_as_noncomparable     41
duplicate_ids                                                0
duplicate_numbers                                            0
missing_overall_scores                                       0
missing_section_sets                                         0


## Results

The headline comparison uses paired debate-level totals. Section scores are a nested corroborating view, not 889 independent debates.

In [3]:
primary = results['primary']
sections = results['sections']
ci_low, ci_high = primary['mean_margin_bootstrap_95_ci']
print(f"Theistic/religious mean: {primary['theist_mean']:.2f}")
print(f"Non-theistic mean:      {primary['non_theist_mean']:.2f}")
print(f"Mean paired margin:     {primary['mean_margin']:5.2f} (bootstrap 95% interval {ci_low:.2f} to {ci_high:.2f})")
print(f"Debate outcomes:         {primary['non_theist_wins']} non-theistic wins / {primary['ties']} ties / {primary['theist_wins']} theistic wins")
print(f"Section outcomes:        {sections['non_theist_higher']} non-theistic higher / {sections['ties']} ties / {sections['theist_higher']} theistic higher")

Theistic/religious mean: 77.46
Non-theistic mean:      83.79
Mean paired margin:      6.34 (bootstrap 95% interval 5.40 to 7.27)
Debate outcomes:         144 non-theistic wins / 5 ties / 20 theistic wins
Section outcomes:        674 non-theistic higher / 33 ties / 182 theistic higher


In [4]:
robustness = pd.DataFrame(results['robustness_rows'])
robustness['mean_margin'] = robustness['mean_margin'].round(2)
print(robustness.to_string(index=False))

                                         check          unit   n  mean_margin  non_theist_wins  ties  theist_wins
                     Primary classified corpus       debates 169         6.34              144     5           20
              One mean per unique speaker pair speaker pairs 160         6.22              136     4           20
Exclude four most frequent non-theist speakers       debates 101         5.05               83     2           16
         Theistic/religious side stored as pro       debates 150         6.73              130     5           15
         Theistic/religious side stored as con       debates  19         3.26               14     0            5


## Takeaways

**Yes—the published assessments contain a large and very consistent non-theistic advantage.** The raw difference is 6.34 points, appears in 144 of 169 comparable debates, and remains positive after limiting repeated speaker pairs, removing the four most frequent non-theist speakers, and looking only at the 19 debates where the theistic/religious side is stored as `con`.

**The safe claim is corpus-specific.** SLUGFESTER selected these debates; repeat speakers are common; the critical side is usually `con`; and the assessment system is AI-assisted. These facts do not erase the pattern, but they prevent treating it as an unbiased estimate of all theist and non-theist debate performance—or as evidence that either worldview is true.

In [5]:
assert primary['non_theist_wins'] + primary['ties'] + primary['theist_wins'] == primary['n']
assert sections['non_theist_higher'] + sections['ties'] + sections['theist_higher'] == sections['section_count']
assert results['orientation']['theist_is_pro']['mean_margin'] > 0
assert results['orientation']['theist_is_con']['mean_margin'] > 0
assert results['concentration']['without_top_four_non_theist_speakers']['mean_margin'] > 0
print(f"Validation assessment: {results['validation']['overall_assessment']}")
print('All high-impact arithmetic and corpus integrity assertions passed.')

Validation assessment: Share with caveats
All high-impact arithmetic and corpus integrity assertions passed.
